In [1]:
!pip install nltk gensim sentencepiece transformers tokenizers pandas numpy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 18.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec, FastText
import gensim.downloader as api

from transformers import BertTokenizer

import sentencepiece as spm

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
df = pd.read_csv("/content/IMDB Dataset.csv")

df.head()

In [ ]:
df = df.sample(5000, random_state=42)

In [ ]:
print(df.columns)

In [ ]:
TEXT_COLUMN = "review"

In [ ]:
text = " ".join(df[TEXT_COLUMN].astype(str).head(100))

In [ ]:
text_lower = text.lower()

print(text_lower[:500])

In [ ]:
sentences = sent_tokenize(text_lower)

print("Number of sentences:", len(sentences))
print(sentences[:5])

In [ ]:
stop_words = set(stopwords.words('english'))

processed_sentences = []

for sentence in sentences:
    words = word_tokenize(sentence)

    filtered = [
        word for word in words
        if word.isalpha() and word not in stop_words
    ]

    processed_sentences.append(filtered)

processed_sentences[:3]

In [ ]:
stemmer = PorterStemmer()

stemming_results = []

for word in processed_sentences[0][:20]:
    stemming_results.append(
        (word, stemmer.stem(word))
    )

pd.DataFrame(
    stemming_results,
    columns=["Original","Stemmed"]
)

In [ ]:
lemmatizer = WordNetLemmatizer()

lemmatization_results = []

for word in processed_sentences[0][:20]:
    lemmatization_results.append(
        (word, lemmatizer.lemmatize(word))
    )

pd.DataFrame(
    lemmatization_results,
    columns=["Original","Lemmatized"]
)

In [ ]:
word2vec_model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [ ]:
embedding_matrix = word2vec_model.wv.vectors

print("Embedding Matrix Shape:")
print(embedding_matrix.shape)

In [ ]:
embedding_matrix[:5]

In [ ]:
word = word2vec_model.wv.index_to_key[0]

print("Word:", word)

print(
    word2vec_model.wv[word]
)

In [ ]:
word2vec_model.wv.most_similar(
    positive=[word],
    topn=5
)

In [ ]:
glove = api.load(
    "glove-wiki-gigaword-100"
)

In [ ]:
glove['computer'][:10]

In [ ]:
fasttext_model = FastText(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [ ]:
fasttext_model.wv["computer"][:10]

In [ ]:
comparison_word = "computer"

results = {
    "Word2Vec":
        word2vec_model.wv[comparison_word][:10]
        if comparison_word in word2vec_model.wv
        else "Not Found",

    "GloVe":
        glove[comparison_word][:10]
        if comparison_word in glove
        else "Not Found",

    "FastText":
        fasttext_model.wv[comparison_word][:10]
}

In [ ]:
results

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

bpe_tokenizer = Tokenizer(BPE())
bpe_tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=1000
)

corpus = [
    " ".join(
        [" ".join(x)
         for x in processed_sentences]
    )
]

bpe_tokenizer.train_from_iterator(
    corpus,
    trainer
)

In [ ]:
sentence = "unhappiness is increasing rapidly"

bpe_output = bpe_tokenizer.encode(sentence)

print(bpe_output.tokens)

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

sentence = "unhappiness is increasing rapidly"

wordpiece_tokens = bert_tokenizer.tokenize(
    sentence
)

print(wordpiece_tokens)

In [ ]:
with open("corpus.txt","w",
          encoding="utf8") as f:

    for sentence in sentences:
        f.write(sentence + "\n")

In [ ]:
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='spm',
    vocab_size=1000
)

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("spm.model")

In [ ]:
sentencepiece_tokens = sp.encode(
    "unhappiness is increasing rapidly",
    out_type=str
)

print(sentencepiece_tokens)

In [ ]:
comparison = pd.DataFrame({
    "BPE":[str(bpe_output.tokens)],
    "WordPiece":[str(wordpiece_tokens)],
    "SentencePiece":[str(sentencepiece_tokens)]
})

comparison

In [ ]:
word = "computer"

comparison_embeddings = pd.DataFrame({
    "Word2Vec":
        word2vec_model.wv[word][:10]
        if word in word2vec_model.wv
        else np.zeros(10),

    "GloVe":
        glove[word][:10]
        if word in glove
        else np.zeros(10),

    "FastText":
        fasttext_model.wv[word][:10]
})

In [ ]:
comparison_embeddings

In [ ]:
print("""
1. Text preprocessing completed.
2. Stopwords removed.
3. Stemming and lemmatization applied.
4. Word2Vec embedding matrix generated.
5. GloVe embeddings loaded.
6. FastText embeddings trained.
7. BPE, WordPiece and SentencePiece compared.
8. Static embedding comparison completed.
""")